In [1]:
# sentiment_scrape.py
# Raw sentiment scraping from Google News, Yahoo Finance, and Reddit.
# Output columns: ['ticker','date','headline','link','domain','is_quality_source','source']

from __future__ import annotations

import os
import time
import json
import math



import hashlib
import logging
import datetime as dt
from typing import Dict, Iterable, List, Optional, Tuple
from urllib.parse import urlparse

import pandas as pd
import requests

# Optional: pip install gnews yfinance pyyaml
try:
    from gnews import GNews
except Exception:
    GNews = None

try:
    import yfinance as yf
except Exception:
    yf = None


# -----------------------------
# Config / Utilities
# -----------------------------
DEFAULT_CONFIG = {
    "quality_domains": [
        "bloomberg.com","ft.com","wsj.com","reuters.com","cnbc.com",
        "marketwatch.com","barrons.com","economist.com","forbes.com",
        "morningstar.com","seekingalpha.com","investors.com","fool.com",
        "finance.yahoo.com","businessinsider.com","investing.com"
    ],
    "reddit": {
        "subreddits": [
            "stocks","investing","wallstreetbets","StockMarket","SecurityAnalysis",
            "finance","FinancialNews","options","Quant"
        ],
        "per_query_limit": 100,        # max per subreddit per month
        "user_agent": "Mozilla/5.0 (compatible; sentiment-scraper/1.0)"
    }
}

In [4]:

def load_config(path: Optional[str]) -> dict:
    if path and os.path.exists(path):
        if yaml is None:
            raise RuntimeError("pyyaml not installed but a YAML config path was provided.")
        with open(path, "r", encoding="utf-8") as f:
            cfg = yaml.safe_load(f) or {}
        # merge shallowly with defaults
        out = DEFAULT_CONFIG.copy()
        out.update({k:v for k,v in (cfg or {}).items() if v is not None})
        # nested reddit merge
        if "reddit" in cfg:
            out["reddit"] = {**DEFAULT_CONFIG["reddit"], **cfg["reddit"]}
        return out
    return DEFAULT_CONFIG

def month_bounds(year: int, month: int) -> Tuple[pd.Timestamp, pd.Timestamp]:
    start = pd.Timestamp(year=year, month=month, day=1, tz="UTC")
    end = (start + pd.offsets.MonthEnd(1)).normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)
    return start, end

def get_domain(url: str) -> str:
    try:
        return urlparse(url).netloc.lower()
    except Exception:
        return ""

def is_quality_source(url: str, quality_domains: List[str]) -> bool:
    d = get_domain(url)
    return any(q in d for q in quality_domains)

def ensure_dt(x) -> pd.Timestamp:
    try:
        ts = pd.to_datetime(x, utc=True, errors="coerce")
        if pd.isna(ts):
            raise ValueError
        return ts
    except Exception:
        return pd.NaT

def md5(s: str) -> str:
    return hashlib.md5(s.encode("utf-8", errors="ignore")).hexdigest()

def safe_sleep(seconds: float):
    try:
        time.sleep(seconds)
    except KeyboardInterrupt:
        pass


# -----------------------------
# Google News
# -----------------------------
def collect_google_news_monthly(
    tickers: Iterable[str],
    asset_map: Optional[Dict[str, str]] = None,
    start_year: int = 2018,
    end_year: Optional[int] = None,
    news_per_month: int = 20,
    language: str = "en",
    country: str = "US",
    config_path: Optional[str] = None,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Requires: pip install gnews
    """
    if GNews is None:
        raise RuntimeError("gnews is not installed. pip install gnews")

    cfg = load_config(config_path)
    quality_domains = cfg["quality_domains"]

    if end_year is None:
        end_year = dt.datetime.utcnow().year

    rows = []
    now = pd.Timestamp.utcnow().normalize()

    for ticker in tickers:
        name = asset_map.get(ticker) if asset_map else None
        search_term = f"{name} stock news" if name else f"{ticker} stock news"
        if verbose:
            print(f"[Google] {ticker} → '{search_term}'")

        for year in range(start_year, end_year + 1):
            for month in range(1, 12 + 1):
                start, end = month_bounds(year, month)
                if start > now:
                    break

                # ask for more, then filter/stop after 'news_per_month' quality items
                googlenews = GNews(language=language, country=country, max_results=news_per_month * 3)
                googlenews.start_date = pd.Timestamp(start.tz_convert(None))
                googlenews.end_date = pd.Timestamp(end.tz_convert(None))

                try:
                    items = googlenews.get_news(search_term) or []
                except Exception as e:
                    if verbose:
                        print(f"   [Google] {ticker} {year}-{month:02d} error: {e}")
                    continue

                quality_count = 0
                for it in items:
                    title = (it.get("title") or "").strip()
                    desc = (it.get("description") or "").strip()
                    link = (it.get("link") or "").strip()
                    headline = (title + " " + desc).strip()
                    if not headline or not link:
                        continue

                    # detect date
                    pub = None
                    for key in ("published date", "publishedAt", "pubDate", "publication_date"):
                        if key in it and it[key]:
                            pub = ensure_dt(it[key])
                            if not pd.isna(pub):
                                break
                    if pd.isna(pub):
                        pub = start  # fallback to month start (UTC)

                    # keep only if in the month bounds (robustness)
                    if not (start <= pub <= end):
                        continue

                    domain = get_domain(link)
                    q = is_quality_source(link, quality_domains)

                    rows.append({
                        "ticker": ticker,
                        "date": pub,
                        "headline": headline,
                        "link": link,
                        "domain": domain,
                        "is_quality_source": bool(q),
                        "source": "google"
                    })

                    if q:
                        quality_count += 1
                        if quality_count >= news_per_month:
                            break

                safe_sleep(0.3)

    df = pd.DataFrame(rows)
    return df


# -----------------------------
# Yahoo Finance News (via yfinance)
# -----------------------------
def collect_yahoo_news(
    tickers: Iterable[str],
    asset_map: Optional[Dict[str, str]] = None,
    start: str = "2018-01-01",
    end: Optional[str] = None,
    config_path: Optional[str] = None,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Requires: pip install yfinance
    Uses yf.Ticker(t).news (providerPublishTime in seconds).
    """
    if yf is None:
        raise RuntimeError("yfinance is not installed. pip install yfinance")

    cfg = load_config(config_path)
    quality_domains = cfg["quality_domains"]

    start_ts = pd.Timestamp(start, utc=True)
    end_ts = pd.Timestamp.utcnow() if end is None else pd.Timestamp(end, utc=True)

    rows = []
    for ticker in tickers:
        try:
            if verbose:
                print(f"[Yahoo] {ticker}")
            news_list = yf.Ticker(ticker).news or []
        except Exception as e:
            if verbose:
                print(f"   [Yahoo] {ticker} error: {e}")
            continue

        for n in news_list:
            title = (n.get("title") or "").strip()
            link = (n.get("link") or "").strip()
            if not title or not link:
                continue

            # convert unix seconds to UTC timestamp
            pub = ensure_dt(pd.to_datetime(n.get("providerPublishTime", None), unit="s", utc=True))
            if pd.isna(pub):
                continue

            if not (start_ts <= pub <= end_ts):
                continue

            domain = get_domain(link)
            q = is_quality_source(link, quality_domains)

            rows.append({
                "ticker": ticker,
                "date": pub,
                "headline": title,      # yahoo usually has strong titles; no desc available here
                "link": link,
                "domain": domain,
                "is_quality_source": bool(q),
                "source": "yahoo"
            })

        safe_sleep(0.15)

    return pd.DataFrame(rows)


# -----------------------------
# Reddit (unauthenticated JSON)
# -----------------------------
def collect_reddit_monthly(
    tickers: Iterable[str],
    asset_map: Optional[Dict[str, str]] = None,
    start_year: int = 2018,
    end_year: Optional[int] = None,
    per_query_limit: Optional[int] = None,
    config_path: Optional[str] = None,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Unauthenticated Reddit JSON search (no PRAW). Rate-limited & simple.
    For each (subreddit, month, term) fetch up to per_query_limit posts sorted by new
    and filter by created_utc ∈ [month_start, month_end].
    """
    cfg = load_config(config_path)
    subs = cfg["reddit"]["subreddits"]
    ua = cfg["reddit"]["user_agent"]
    default_limit = cfg["reddit"]["per_query_limit"]
    limit = per_query_limit or default_limit

    headers = {"User-Agent": ua}
    if end_year is None:
        end_year = dt.datetime.utcnow().year

    rows = []
    now = pd.Timestamp.utcnow().normalize()

    def search_subreddit(sub: str, query: str, after: Optional[str] = None) -> dict:
        url = f"https://www.reddit.com/r/{sub}/search.json"
        params = {
            "q": query,
            "restrict_sr": 1,
            "sort": "new",
            "limit": 100,
            "after": after or ""
        }
        r = requests.get(url, params=params, headers=headers, timeout=20)
        r.raise_for_status()
        return r.json()

    for ticker in tickers:
        name = asset_map.get(ticker) if asset_map else None
        # keep query simple to reduce false positives
        # prioritize company name if present, else use ticker
        term = f'"{name}"' if name else ticker
        if verbose:
            print(f"[Reddit] {ticker} → term={term} in {len(subs)} subs")

        for year in range(start_year, end_year + 1):
            for month in range(1, 12 + 1):
                start, end = month_bounds(year, month)
                if start > now:
                    break

                month_keep = 0
                for sub in subs:
                    after = None
                    # Paginate until we either surpass limit or we are outside month window
                    while month_keep < limit:
                        try:
                            data = search_subreddit(sub, term, after)
                        except Exception as e:
                            if verbose:
                                print(f"   [Reddit] {sub} {year}-{month:02d} error: {e}")
                            break

                        children = (data.get("data") or {}).get("children") or []
                        if not children:
                            break

                        for ch in children:
                            d = ch.get("data") or {}
                            title = (d.get("title") or "").strip()
                            selftext = (d.get("selftext") or "").strip()
                            url = (d.get("url") or "").strip()
                            created = ensure_dt(pd.to_datetime(d.get("created_utc", None), unit="s", utc=True))
                            if pd.isna(created):
                                continue

                            # stop paginating if everything is now older than month
                            if created < start:
                                children = []  # to stop outer while
                                break

                            # only keep posts inside the month
                            if start <= created <= end:
                                headline = (title + " " + selftext).strip() if selftext else title
                                domain = get_domain(url) if url else "reddit.com"
                                q = domain == "reddit.com" or domain.endswith("reddit.com")

                                rows.append({
                                    "ticker": ticker,
                                    "date": created,
                                    "headline": headline,
                                    "link": f"https://www.reddit.com{d.get('permalink','')}".strip(),
                                    "domain": domain if domain else "reddit.com",
                                    "is_quality_source": bool(q),  # mark reddit as non-premium but traceable
                                    "source": "reddit"
                                })
                                month_keep += 1
                                if month_keep >= limit:
                                    break

                        # move to next page
                        after = (data.get("data") or {}).get("after")
                        if not after or not children:
                            break

                        safe_sleep(0.4)

                safe_sleep(0.2)

    return pd.DataFrame(rows)


# -----------------------------
# Unification + Post-processing
# -----------------------------
def unify_and_clean(dfs: List[pd.DataFrame]) -> pd.DataFrame:
    cols = ["ticker","date","headline","link","domain","is_quality_source","source"]
    frames = []
    for df in dfs:
        if df is None or df.empty:
            continue
        keep = [c for c in cols if c in df.columns]
        tmp = df[keep].copy()
        # enforce dtypes
        if "date" in tmp.columns:
            tmp["date"] = pd.to_datetime(tmp["date"], utc=True, errors="coerce")
        for b in ("is_quality_source",):
            if b in tmp.columns:
                tmp[b] = tmp[b].fillna(False).astype(bool)
        frames.append(tmp)

    if not frames:
        return pd.DataFrame(columns=cols)

    out = pd.concat(frames, ignore_index=True)
    # drop rows with no text/link/ticker/date
    out = out.dropna(subset=["ticker","date","headline"], how="any")
    # deduplicate by link (primary) and fallback on (ticker, date_day, md5(headline))
    out["date_day"] = out["date"].dt.floor("D")
    out["_sig"] = out["link"].fillna("") + "|" + out["ticker"] + "|" + out["date_day"].astype(str)
    # Prefer link-based drop first
    if "link" in out.columns:
        out = out.drop_duplicates(subset=["link"])
    # extra safeguard on signature of (ticker,day,text)
    out["_sig2"] = out.apply(lambda r: r["ticker"] + "|" + str(r["date_day"]) + "|" + md5(r["headline"]), axis=1)
    out = out.drop_duplicates(subset=["_sig2"]).drop(columns=["_sig","_sig2","date_day"])

    # sort
    out = out.sort_values("date").reset_index(drop=True)
    return out


# -----------------------------
# Public entry point
# -----------------------------
def collect_raw_sentiment_data(
    tickers: Iterable[str],
    start_year: int = 2018,
    end_year: Optional[int] = None,
    asset_map: Optional[Dict[str, str]] = None,
    config_path: Optional[str] = None,
    news_per_month_google: int = 20,
    reddit_per_query_limit: Optional[int] = None,
    yahoo_range: Tuple[str, Optional[str]] = ("2018-01-01", None),
    sources: Tuple[str, ...] = ("google","yahoo","reddit"),
    verbose: bool = True
) -> pd.DataFrame:

    dfs = []



    if "google" in sources:
        df_g = collect_google_news_monthly(
            tickers=tickers,
            asset_map=asset_map,
            start_year=start_year,
            end_year=end_year,
            news_per_month=news_per_month_google,
            config_path=config_path,
            verbose=verbose
        )
        dfs.append(df_g)

    if "yahoo" in sources:
        start, end = yahoo_range
        df_y = collect_yahoo_news(
            tickers=tickers,
            asset_map=asset_map,
            start=start,
            end=end,
            config_path=config_path,
            verbose=verbose
        )
        dfs.append(df_y)


    if "reddit" in sources:
        df_r = collect_reddit_monthly(
            tickers=tickers,
            asset_map=asset_map,
            start_year=start_year,
            end_year=end_year,
            per_query_limit=reddit_per_query_limit,
            config_path=config_path,
            verbose=verbose
        )
        dfs.append(df_r)

    out = unify_and_clean(dfs)
    # Final schema guarantee and ordering
    wanted = ["ticker","date","headline","link","domain","is_quality_source","source"]
    for c in wanted:
        if c not in out.columns:
            out[c] = pd.Series(dtype="object" if c not in ("date","is_quality_source") else ("datetime64[ns, UTC]" if c=="date" else "bool"))
    return out[wanted]

In [7]:

# Example inputs
tickers = ["AAPL","NVDA","MSFT"]
asset_map = {"AAPL":"Apple","NVDA":"NVIDIA","MSFT":"Microsoft"}

df_raw = collect_raw_sentiment_data(
    tickers=tickers,
    start_year=2024,
    end_year=2024,
    asset_map=asset_map,
    config_path=None,  # or None to use built-in defaults
    news_per_month_google=20,
    reddit_per_query_limit=50,
    yahoo_range=("2020-01-01", None),
    sources=("google"),
    verbose=True
)

print(df_raw.head())
print(df_raw.source.value_counts())


[Google] AAPL → 'Apple stock news'
[Google] NVDA → 'NVIDIA stock news'
[Google] MSFT → 'Microsoft stock news'
Empty DataFrame
Columns: [ticker, date, headline, link, domain, is_quality_source, source]
Index: []
Series([], Name: count, dtype: int64)
